# **Learn, Love, Live**

In [1]:
print("jai maa kali")

jai maa kali


In [2]:
# understand or working with dataframe modules
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Display settings for pandas DataFrames
pd.set_option('display.max_columns', None)  # Show all columns
# pd.set_option('display.max_rows', None)     # Show all rows
# pd.set_option('display.max_colwidth', None) # Show full column content
# pd.set_option('display.width', None)        # Auto-detect terminal width
# pd.set_option('display.expand_frame_repr', False) # Don't wrap to multiple pages

## **Start**

In [3]:
gym = pd.read_csv("data/model_data/list_execise.csv")
gym.head(1)

,id,name,bodyPart,target,muscle_group,equipment,instructions,tutorial_url,icon,description,difficulty level,difficulty reason
0,1,3/4 sit-up,waist,abs,"hip flexors, lower back",body weight,Lie flat on your back with your knees bent and...,https://v2.exercisedb.io/image/MOnK4iG0MEt9h8,NaN,The 3/4 sit-up is a core exercise that targets...,Beginner,This exercise primarily uses bodyweight and in...


In [4]:
gym.columns

Index(['id', 'name', 'bodyPart', 'target', 'muscle_group', 'equipment',
       'instructions', 'tutorial_url', 'icon', 'description',
       'difficulty level', 'difficulty reason'],
      dtype='object')

In [5]:
# ['id', 'name', 'bodyPart', 'target', 'muscle_group', 'equipment','instructions', 'tutorial_url', 'ion', 'description',
#     'difficulty level', 'difficulty reason']


    # id = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    # name = Column(String(255), nullable=False, index=True)
    # muscle_group = Column(String(100))  # chest, back, legs, shoulders, arms, core
    # equipment = Column(String(100))  # barbell, dumbbell, machine, bodyweight, cable
    # difficulty_level = Column(String(50))  # beginner, intermediate, advanced
    # description = Column(Text)
    # tutorial_url = Column(String(500))
    # is_active = Column(Boolean, default=True)
    # created_at = Column(DateTime(timezone=True), server_default=func.now())
    # updated_at = Column(DateTime(timezone=True), onupdate=func.now())

In [6]:
# Weight increase:
# - Beginner: +2.5%
# - Intermediate: +5%
# - Advanced: +2.5%

# Reps increase:
# - +1–2 reps per set

# Sets increase:
# - +1 set (max)


###  **Python script for converting your CSV to PostgreSQL database**

In [3]:
list_exe = pd.read_csv("data/model_data/list_execise.csv")
list_exe.head(1)

,id,name,bodyPart,target,muscle_group,equipment,instructions,tutorial_url,icon,description,difficulty level,difficulty reason
0,1,3/4 sit-up,waist,abs,"hip flexors, lower back",body weight,Lie flat on your back with your knees bent and...,https://v2.exercisedb.io/image/MOnK4iG0MEt9h8,NaN,The 3/4 sit-up is a core exercise that targets...,Beginner,This exercise primarily uses bodyweight and in...


In [4]:
list_exe.columns

Index(['id', 'name', 'bodyPart', 'target', 'muscle_group', 'equipment',
       'instructions', 'tutorial_url', 'icon', 'description',
       'difficulty level', 'difficulty reason'],
      dtype='object')

In [10]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from io import StringIO
import time
import uuid

def ultra_fast_csv_to_postgres(csv_path, table_name="exercises"):
    """
    Ultra-fast CSV to PostgreSQL using COPY command
    Fixed for UUID conversion
    """
    
    print("⚡ ULTRA-FAST CSV IMPORT")
    print("="*60)
    
    start_time = time.time()
    
    # Database config
    db_config = {
        'host': 'localhost',
        'database': 'GoLift_db',
        'user': 'postgres',
        'password': 'kali7',
        'port': '5432'
    }
    
    try:
        # Step 1: Load CSV
        csv_start = time.time()
        df = pd.read_csv(csv_path)
        csv_time = time.time() - csv_start
        print(f"✅ CSV loaded: {len(df)} rows in {csv_time:.2f}s")
        
        # Step 2: Rename columns
        df = df.rename(columns={
            'bodyPart': 'body_part',
            'target': 'target_muscle',
            'difficulty level': 'difficulty_level',
            'difficulty reason': 'difficulty_reason'
        })
        
        # Step 3: Convert numeric IDs to UUID format
        print("🔄 Converting IDs to UUID format...")
        id_start = time.time()
        
        # Option 1: Generate consistent UUIDs from numeric IDs
        def numeric_id_to_uuid(numeric_id):
            """Convert numeric ID to deterministic UUID"""
            # Create UUID from namespace and numeric ID
            # This ensures same numeric ID always produces same UUID
            namespace = uuid.UUID('6ba7b810-9dad-11d1-80b4-00c04fd430c8')
            return str(uuid.uuid5(namespace, str(numeric_id)))
        
        # Apply conversion if 'id' column exists and contains numbers
        if 'id' in df.columns:
            # Check if IDs are numeric
            try:
                # Try to convert to numeric
                df['id'] = pd.to_numeric(df['id'])
                # Convert to UUID
                df['id'] = df['id'].apply(numeric_id_to_uuid)
                print(f"  Converted {len(df)} numeric IDs to UUIDs")
            except:
                # If already strings or UUIDs, leave as is
                print(f"  IDs are already string/UUID format")
        
        id_time = time.time() - id_start
        print(f"✅ ID conversion: {id_time:.2f}s")
        
        # Step 4: Connect and use COPY
        copy_start = time.time()
        
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Create temporary staging table
        staging_table = f"staging_{table_name}_{int(time.time())}"
        
        # Create staging table with TEXT columns (for flexible import)
        cursor.execute(sql.SQL("""
            CREATE TEMP TABLE {staging_table} (
                id TEXT,
                name TEXT,
                body_part TEXT,
                target_muscle TEXT,
                muscle_group TEXT,
                equipment TEXT,
                instructions TEXT,
                tutorial_url TEXT,
                icon TEXT,
                description TEXT,
                difficulty_level TEXT,
                difficulty_reason TEXT
            ) ON COMMIT DROP
        """).format(staging_table=sql.Identifier(staging_table)))
        
        # Prepare data for COPY
        output = StringIO()
        
        # Write to CSV format without headers
        # We'll use \t as delimiter and \\N for NULL
        for _, row in df.iterrows():
            # Escape tabs and newlines
            row_str = []
            for val in row:
                if pd.isna(val):
                    row_str.append('\\N')  # PostgreSQL NULL
                else:
                    # Convert to string and escape special characters
                    val_str = str(val)
                    val_str = val_str.replace('\\', '\\\\')
                    val_str = val_str.replace('\t', '\\t')
                    val_str = val_str.replace('\n', '\\n')
                    val_str = val_str.replace('\r', '\\r')
                    row_str.append(val_str)
            output.write('\t'.join(row_str) + '\n')
        
        output.seek(0)
        
        # Copy to staging table
        cursor.copy_from(output, staging_table, null='\\N')
        print(f"✅ Data copied to staging table")
        
        # Step 5: Backup existing data
        backup_start = time.time()
        timestamp = int(time.time())
        backup_table = f"{table_name}_backup_{timestamp}"
        
        cursor.execute(sql.SQL("""
            CREATE TABLE {backup_table} AS 
            SELECT * FROM {table_name} WHERE 1=0
        """).format(
            backup_table=sql.Identifier(backup_table),
            table_name=sql.Identifier(table_name)
        ))
        
        # Copy existing data to backup
        cursor.execute(sql.SQL("""
            INSERT INTO {backup_table}
            SELECT * FROM {table_name}
        """).format(
            backup_table=sql.Identifier(backup_table),
            table_name=sql.Identifier(table_name)
        ))
        backup_time = time.time() - backup_start
        print(f"✅ Backup created: {backup_table} ({cursor.rowcount} rows)")
        
        # Step 6: Clear and insert from staging
        insert_start = time.time()
        
        # Clear existing data
        cursor.execute(sql.SQL("TRUNCATE TABLE {table_name} CASCADE").format(
            table_name=sql.Identifier(table_name)
        ))
        
        # Insert from staging with proper type casting
        cursor.execute(sql.SQL("""
            INSERT INTO {table_name} (
                id, name, body_part, target_muscle, muscle_group, equipment,
                instructions, tutorial_url, icon, description, difficulty_level,
                difficulty_reason, created_at
            )
            SELECT 
                s.id::UUID,
                s.name,
                s.body_part,
                s.target_muscle,
                s.muscle_group,
                s.equipment,
                s.instructions,
                s.tutorial_url,
                s.icon,
                s.description,
                s.difficulty_level,
                s.difficulty_reason,
                CURRENT_TIMESTAMP
            FROM {staging_table} s
        """).format(
            table_name=sql.Identifier(table_name),
            staging_table=sql.Identifier(staging_table)
        ))
        
        insert_time = time.time() - insert_start
        inserted_count = cursor.rowcount
        print(f"✅ Inserted {inserted_count} rows")
        
        # Clean up staging table (will be dropped automatically due to TEMP)
        
        conn.commit()
        cursor.close()
        conn.close()
        
        copy_time = time.time() - copy_start
        
        total_time = time.time() - start_time
        
        print("\n" + "="*60)
        print("📈 PERFORMANCE RESULTS")
        print("="*60)
        print(f"📥 CSV Load:          {csv_time:.3f} seconds")
        print(f"🔄 ID Conversion:     {id_time:.3f} seconds")
        print(f"💾 Backup:            {backup_time:.3f} seconds")
        print(f"📤 Data Insert:       {insert_time:.3f} seconds")
        print("-"*60)
        print(f"🚀 TOTAL TIME:        {total_time:.2f} seconds")
        print(f"📊 Rows processed:    {inserted_count}")
        print(f"📈 Speed:             {inserted_count/total_time:.0f} rows/second")
        
        # Verify
        if inserted_count == len(df):
            print("✅ SUCCESS: All rows inserted correctly!")
        else:
            print(f"⚠️  WARNING: Expected {len(df)} rows, got {inserted_count}")
        
        print(f"💾 Backup table: {backup_table}")
        
        return df, total_time
        
    except Exception as e:
        print(f"❌ Error: {e}")
        # Add more detailed error info
        import traceback
        traceback.print_exc()
        raise

# Run the function
df, total_time = ultra_fast_csv_to_postgres("data/model_data/list_execise.csv")

⚡ ULTRA-FAST CSV IMPORT
✅ CSV loaded: 1324 rows in 0.61s
🔄 Converting IDs to UUID format...
  Converted 1324 numeric IDs to UUIDs
✅ ID conversion: 0.05s
✅ Data copied to staging table
✅ Backup created: exercises_backup_1767501973 (1324 rows)
✅ Inserted 1324 rows

📈 PERFORMANCE RESULTS
📥 CSV Load:          0.611 seconds
🔄 ID Conversion:     0.052 seconds
💾 Backup:            0.081 seconds
📤 Data Insert:       0.358 seconds
------------------------------------------------------------
🚀 TOTAL TIME:        2.87 seconds
📊 Rows processed:    1324
📈 Speed:             462 rows/second
✅ SUCCESS: All rows inserted correctly!
💾 Backup table: exercises_backup_1767501973


### **Python script with PostgreSQL queries that you can run in your Jupyter notebook to upload this workout data to your database**

In [30]:
# import pandas as pd
# import uuid
# from datetime import datetime
# import os|

# def generate_weight_loss_workout():
#     """
#     Generate weight loss workout plan
#     """
#     workout_days = [
#         {"day_name": "Day 1: HIIT Cardio + Full Body", "day_number": 1, "workout_type": "workout"},
#         {"day_name": "Day 2: Cardio + Core", "day_number": 2, "workout_type": "workout"},
#         {"day_name": "Day 3: Active Recovery", "day_number": 3, "workout_type": "recovery"},
#         {"day_name": "Day 4: Strength & Cardio Combo", "day_number": 4, "workout_type": "workout"},
#         {"day_name": "Day 5: Endurance Cardio", "day_number": 5, "workout_type": "workout"},
#         {"day_name": "Day 6: Active Rest", "day_number": 6, "workout_type": "recovery"},
#         {"day_name": "Day 7: Active Rest", "day_number": 7, "workout_type": "recovery"}
#     ]
    
#     workout_exercises = {
#         1: {
#             "warmup": [
#                 {"name": "Jump Rope", "sets": 3, "reps": "60 sec", "rest": 20},
#                 {"name": "High Knees", "sets": 2, "reps": "45 sec", "rest": 20},
#                 {"name": "Dynamic Stretching", "sets": 2, "reps": "30 sec each", "rest": 15}
#             ],
#             "main": [
#                 {"name": "Burpees", "sets": 4, "reps": 15, "rest": 45},
#                 {"name": "Kettlebell Swings", "sets": 4, "reps": 20, "rest": 45},
#                 {"name": "Mountain Climbers", "sets": 4, "reps": "30 sec", "rest": 45},
#                 {"name": "Box Jumps", "sets": 4, "reps": 12, "rest": 45},
#                 {"name": "Battle Ropes", "sets": 4, "reps": "30 sec", "rest": 45}
#             ],
#             "relax": [
#                 {"name": "Standing Quad Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Hamstring Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Deep Breathing", "sets": 1, "reps": "2 min", "rest": 0}
#             ]
#         },
#         2: {
#             "warmup": [
#                 {"name": "Torso Twists", "sets": 2, "reps": 20, "rest": 20},
#                 {"name": "Leg Swings", "sets": 2, "reps": "15 each", "rest": 20},
#                 {"name": "Arm Circles", "sets": 2, "reps": "30 sec", "rest": 15}
#             ],
#             "main": [
#                 {"name": "Running/Sprints", "sets": 1, "reps": "20 min intervals", "rest": 0},
#                 {"name": "Bicycle Crunches", "sets": 4, "reps": 25, "rest": 45},
#                 {"name": "Russian Twists", "sets": 4, "reps": "20 each side", "rest": 45},
#                 {"name": "Plank Variations", "sets": 4, "reps": "45 sec each", "rest": 45},
#                 {"name": "Leg Raises", "sets": 4, "reps": 20, "rest": 45}
#             ],
#             "relax": [
#                 {"name": "Cobra Stretch", "sets": 1, "reps": "30 sec", "rest": 0},
#                 {"name": "Child's Pose", "sets": 1, "reps": "60 sec", "rest": 0},
#                 {"name": "Cat-Cow Stretch", "sets": 2, "reps": 10, "rest": 0}
#             ]
#         },
#         4: {
#             "warmup": [
#                 {"name": "Jumping Jacks", "sets": 3, "reps": "45 sec", "rest": 20},
#                 {"name": "Bodyweight Squats", "sets": 2, "reps": 20, "rest": 20},
#                 {"name": "Arm Swings", "sets": 2, "reps": 15, "rest": 15}
#             ],
#             "main": [
#                 {"name": "Squat to Press", "sets": 4, "reps": 15, "rest": 45},
#                 {"name": "Renegade Rows", "sets": 4, "reps": "12 each side", "rest": 45},
#                 {"name": "Jump Lunges", "sets": 4, "reps": 16, "rest": 45},
#                 {"name": "Push-up to Plank Jack", "sets": 4, "reps": 12, "rest": 45},
#                 {"name": "Skater Jumps", "sets": 4, "reps": 20, "rest": 45}
#             ],
#             "relax": [
#                 {"name": "Hip Flexor Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Chest Opener", "sets": 1, "reps": "30 sec", "rest": 0},
#                 {"name": "Standing Forward Fold", "sets": 1, "reps": "60 sec", "rest": 0}
#             ]
#         },
#         5: {
#             "warmup": [
#                 {"name": "Light Jog", "sets": 1, "reps": "5 min", "rest": 0},
#                 {"name": "Dynamic Leg Swings", "sets": 2, "reps": "15 each", "rest": 20},
#                 {"name": "Shoulder Rolls", "sets": 2, "reps": 10, "rest": 15}
#             ],
#             "main": [
#                 {"name": "Cycling/Rowing", "sets": 1, "reps": "30 min steady", "rest": 0},
#                 {"name": "Jump Squats", "sets": 4, "reps": 15, "rest": 45},
#                 {"name": "Bear Crawls", "sets": 4, "reps": "30 sec", "rest": 45},
#                 {"name": "Side Plank Dips", "sets": 4, "reps": "15 each side", "rest": 45},
#                 {"name": "Wall Sits", "sets": 4, "reps": "60 sec", "rest": 45}
#             ],
#             "relax": [
#                 {"name": "Figure-4 Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Upper Back Stretch", "sets": 1, "reps": "30 sec", "rest": 0},
#                 {"name": "Seated Meditation", "sets": 1, "reps": "3 min", "rest": 0}
#             ]
#         }
#     }
    
#     return generate_workout_dataframe(workout_days, workout_exercises, "Weight Loss")

# def generate_sixpack_workout():
#     """
#     Generate six-pack abs workout plan
#     """
#     workout_days = [
#         {"day_name": "Day 1: Core Focus + Cardio", "day_number": 1, "workout_type": "workout"},
#         {"day_name": "Day 2: Full Body + Core Activation", "day_number": 2, "workout_type": "workout"},
#         {"day_name": "Day 3: Active Recovery", "day_number": 3, "workout_type": "recovery"},
#         {"day_name": "Day 4: Obliques & Anti-Rotation", "day_number": 4, "workout_type": "workout"},
#         {"day_name": "Day 5: Lower Abs & Compound", "day_number": 5, "workout_type": "workout"},
#         {"day_name": "Day 6: Recovery & Light Core", "day_number": 6, "workout_type": "recovery"},
#         {"day_name": "Day 7: Recovery & Light Core", "day_number": 7, "workout_type": "recovery"}
#     ]
    
#     workout_exercises = {
#         1: {
#             "warmup": [
#                 {"name": "Cat-Cow Stretch", "sets": 2, "reps": 10, "rest": 20},
#                 {"name": "Bird-Dog", "sets": 2, "reps": "10 each side", "rest": 20},
#                 {"name": "Dead Bugs", "sets": 2, "reps": 10, "rest": 20}
#             ],
#             "main": [
#                 {"name": "Hanging Leg Raises", "sets": 4, "reps": 15, "rest": 60},
#                 {"name": "Ab Wheel Rollouts", "sets": 4, "reps": 12, "rest": 60},
#                 {"name": "Cable Crunches", "sets": 4, "reps": 20, "rest": 60},
#                 {"name": "L-Sit Holds", "sets": 4, "reps": "30 sec", "rest": 60},
#                 {"name": "Dragon Flags", "sets": 4, "reps": 8, "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Cobra Stretch", "sets": 1, "reps": "30 sec", "rest": 0},
#                 {"name": "Seated Forward Fold", "sets": 1, "reps": "60 sec", "rest": 0},
#                 {"name": "Supine Twist", "sets": 1, "reps": "30 sec each", "rest": 0}
#             ]
#         },
#         2: {
#             "warmup": [
#                 {"name": "Plank to Down Dog", "sets": 2, "reps": 10, "rest": 20},
#                 {"name": "Side Plank Hip Dips", "sets": 2, "reps": "10 each", "rest": 20},
#                 {"name": "High Plank Shoulder Taps", "sets": 2, "reps": 20, "rest": 20}
#             ],
#             "main": [
#                 {"name": "Deadlifts", "sets": 4, "reps": 8, "rest": 90},
#                 {"name": "Overhead Press", "sets": 4, "reps": 10, "rest": 90},
#                 {"name": "Pull-ups", "sets": 4, "reps": "max", "rest": 90},
#                 {"name": "Pallof Press", "sets": 4, "reps": "12 each side", "rest": 60},
#                 {"name": "Windshield Wipers", "sets": 4, "reps": "10 each side", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Child's Pose with Side Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Pigeon Pose", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Diaphragmatic Breathing", "sets": 1, "reps": "3 min", "rest": 0}
#             ]
#         },
#         4: {
#             "warmup": [
#                 {"name": "Russian Twist (light)", "sets": 2, "reps": 20, "rest": 20},
#                 {"name": "Side Bend Stretch", "sets": 2, "reps": "10 each", "rest": 20},
#                 {"name": "Torso Rotations", "sets": 2, "reps": "15 each way", "rest": 20}
#             ],
#             "main": [
#                 {"name": "Side Plank Raises", "sets": 4, "reps": "15 each side", "rest": 60},
#                 {"name": "Woodchoppers", "sets": 4, "reps": "12 each side", "rest": 60},
#                 {"name": "Bicycle Crunches", "sets": 4, "reps": 30, "rest": 60},
#                 {"name": "Hanging Knee Tucks with Twist", "sets": 4, "reps": "12 each side", "rest": 60},
#                 {"name": "Landmine 180s", "sets": 4, "reps": "10 each side", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Seated Spinal Twist", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Happy Baby Pose", "sets": 1, "reps": "60 sec", "rest": 0},
#                 {"name": "Supine Figure-4", "sets": 1, "reps": "30 sec each", "rest": 0}
#             ]
#         },
#         5: {
#             "warmup": [
#                 {"name": "Reverse Crunches", "sets": 2, "reps": 15, "rest": 20},
#                 {"name": "Leg Lowering", "sets": 2, "reps": 10, "rest": 20},
#                 {"name": "Scissor Kicks", "sets": 2, "reps": 20, "rest": 20}
#             ],
#             "main": [
#                 {"name": "Front Squats", "sets": 4, "reps": 8, "rest": 90},
#                 {"name": "Push-ups with Rotation", "sets": 4, "reps": "12 each side", "rest": 60},
#                 {"name": "Plank Variations (RKC, Side, Reverse)", "sets": 4, "reps": "45 sec each", "rest": 60},
#                 {"name": "V-Ups", "sets": 4, "reps": 15, "rest": 60},
#                 {"name": "Vacuum Holds", "sets": 4, "reps": "30 sec", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Supine Hamstring Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Prayer Stretch", "sets": 1, "reps": "60 sec", "rest": 0},
#                 {"name": "Deep Core Breathing", "sets": 1, "reps": "3 min", "rest": 0}
#             ]
#         }
#     }
    
#     return generate_workout_dataframe(workout_days, workout_exercises, "Six-Pack Gain")

# def generate_muscle_gain_workout():
#     """
#     Generate muscle gain workout plan (from previous example, updated)
#     """
#     workout_days = [
#         {"day_name": "Day 1: Upper Body Push", "day_number": 1, "workout_type": "workout"},
#         {"day_name": "Day 2: Lower Body", "day_number": 2, "workout_type": "workout"},
#         {"day_name": "Day 3: Active Recovery", "day_number": 3, "workout_type": "recovery"},
#         {"day_name": "Day 4: Upper Body Pull", "day_number": 4, "workout_type": "workout"},
#         {"day_name": "Day 5: Full Body", "day_number": 5, "workout_type": "workout"},
#         {"day_name": "Day 6: Recovery", "day_number": 6, "workout_type": "recovery"},
#         {"day_name": "Day 7: Recovery", "day_number": 7, "workout_type": "recovery"}
#     ]
    
#     workout_exercises = {
#         1: {
#             "warmup": [
#                 {"name": "Arm Circles", "sets": 2, "reps": "30 sec", "rest": 30},
#                 {"name": "Jumping Jacks", "sets": 2, "reps": "30 sec", "rest": 30},
#                 {"name": "Cat-Cow Stretch", "sets": 2, "reps": 10, "rest": 30}
#             ],
#             "main": [
#                 {"name": "Bench Press", "sets": 4, "reps": "8-10", "rest": 90},
#                 {"name": "Overhead Press", "sets": 4, "reps": "8-10", "rest": 90},
#                 {"name": "Incline Dumbbell Press", "sets": 3, "reps": "10-12", "rest": 75},
#                 {"name": "Tricep Dips", "sets": 3, "reps": "10-15", "rest": 60},
#                 {"name": "Lateral Raises", "sets": 3, "reps": "12-15", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Chest Stretch", "sets": 1, "reps": "30 sec", "rest": 0},
#                 {"name": "Tricep Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Shoulder Rolls", "sets": 2, "reps": 10, "rest": 0}
#             ]
#         },
#         2: {
#             "warmup": [
#                 {"name": "Leg Swings", "sets": 2, "reps": "15 each", "rest": 30},
#                 {"name": "Bodyweight Squats", "sets": 2, "reps": 15, "rest": 30},
#                 {"name": "Hip Circles", "sets": 2, "reps": "10 each way", "rest": 30}
#             ],
#             "main": [
#                 {"name": "Barbell Squats", "sets": 4, "reps": "6-8", "rest": 120},
#                 {"name": "Romanian Deadlifts", "sets": 4, "reps": "8-10", "rest": 90},
#                 {"name": "Leg Press", "sets": 3, "reps": "10-12", "rest": 75},
#                 {"name": "Leg Curls", "sets": 3, "reps": "12-15", "rest": 60},
#                 {"name": "Calf Raises", "sets": 4, "reps": "15-20", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Quad Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Hamstring Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Pigeon Pose", "sets": 1, "reps": "30 sec each", "rest": 0}
#             ]
#         },
#         4: {
#             "warmup": [
#                 {"name": "Band Pull-Aparts", "sets": 2, "reps": 15, "rest": 30},
#                 {"name": "Scapular Wall Slides", "sets": 2, "reps": 10, "rest": 30},
#                 {"name": "Neck Tilts", "sets": 2, "reps": "10 each", "rest": 30}
#             ],
#             "main": [
#                 {"name": "Pull-ups", "sets": 4, "reps": "max", "rest": 90},
#                 {"name": "Bent-over Rows", "sets": 4, "reps": "8-10", "rest": 90},
#                 {"name": "Lat Pulldowns", "sets": 3, "reps": "10-12", "rest": 75},
#                 {"name": "Barbell Curls", "sets": 3, "reps": "10-12", "rest": 60},
#                 {"name": "Face Pulls", "sets": 3, "reps": "15-20", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Lat Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Bicep Stretch", "sets": 1, "reps": "30 sec each", "rest": 0},
#                 {"name": "Child's Pose", "sets": 1, "reps": "60 sec", "rest": 0}
#             ]
#         },
#         5: {
#             "warmup": [
#                 {"name": "High Knees", "sets": 2, "reps": "30 sec", "rest": 30},
#                 {"name": "Torso Twists", "sets": 2, "reps": "15 each", "rest": 30},
#                 {"name": "Arm Swings", "sets": 2, "reps": 15, "rest": 30}
#             ],
#             "main": [
#                 {"name": "Deadlifts", "sets": 3, "reps": "5-7", "rest": 120},
#                 {"name": "Dumbbell Lunges", "sets": 3, "reps": "10-12 each", "rest": 75},
#                 {"name": "Push-ups", "sets": 3, "reps": "max", "rest": 60},
#                 {"name": "Seated Cable Rows", "sets": 3, "reps": "10-12", "rest": 75},
#                 {"name": "Plank", "sets": 3, "reps": "60 sec", "rest": 60}
#             ],
#             "relax": [
#                 {"name": "Full Body Stretch", "sets": 1, "reps": "60 sec", "rest": 0},
#                 {"name": "Deep Breathing", "sets": 1, "reps": "2 min", "rest": 0},
#                 {"name": "Legs Up the Wall", "sets": 1, "reps": "5 min", "rest": 0}
#             ]
#         }
#     }
    
#     return generate_workout_dataframe(workout_days, workout_exercises, "Muscle Gain")

# def generate_workout_dataframe(workout_days, workout_exercises, workout_type):
#     """
#     Helper function to generate dataframe for any workout type
#     """
#     data = []
    
#     for day in workout_days:
#         day_num = day["day_number"]
        
#         if day["workout_type"] == "workout" and day_num in workout_exercises:
#             # Add warmup exercises
#             for i, exercise in enumerate(workout_exercises[day_num]["warmup"]):
#                 data.append({
#                     "workout_day": day["day_name"],
#                     "day_number": day_num,
#                     "exercise_name": exercise["name"],
#                     "stage_of_exercises": "warmup",
#                     "sequence_order": i + 1,
#                     "sets": exercise["sets"],
#                     "reps": exercise["reps"],
#                     "weight": "Bodyweight",
#                     "rest_seconds": exercise["rest"],
#                     "notes": f"{workout_type} - Warmup: Focus on form and mobility",
#                     "workout_type": workout_type
#                 })
            
#             # Add main exercises
#             for i, exercise in enumerate(workout_exercises[day_num]["main"]):
#                 weight_note = "Bodyweight" if "min" in str(exercise["reps"]) or "sec" in str(exercise["reps"]) else "Progressive overload"
#                 data.append({
#                     "workout_day": day["day_name"],
#                     "day_number": day_num,
#                     "exercise_name": exercise["name"],
#                     "stage_of_exercises": "main",
#                     "sequence_order": i + 1,
#                     "sets": exercise["sets"],
#                     "reps": exercise["reps"],
#                     "weight": weight_note,
#                     "rest_seconds": exercise["rest"],
#                     "notes": f"{workout_type} - Main: Focus on intensity and form",
#                     "workout_type": workout_type
#                 })
            
#             # Add relax exercises
#             for i, exercise in enumerate(workout_exercises[day_num]["relax"]):
#                 data.append({
#                     "workout_day": day["day_name"],
#                     "day_number": day_num,
#                     "exercise_name": exercise["name"],
#                     "stage_of_exercises": "relax",
#                     "sequence_order": i + 1,
#                     "sets": exercise["sets"],
#                     "reps": exercise["reps"],
#                     "weight": "Bodyweight",
#                     "rest_seconds": exercise["rest"],
#                     "notes": f"{workout_type} - Cool down: Focus on recovery",
#                     "workout_type": workout_type
#                 })
#         else:
#             # Recovery days
#             if workout_type == "Weight Loss":
#                 recovery_activities = [
#                     {"name": "Brisk Walking", "duration": "45 min", "desc": "Low-impact cardio"},
#                     {"name": "Yoga/Stretching", "duration": "30 min", "desc": "Gentle flow and mobility"},
#                     {"name": "Foam Rolling", "duration": "15 min", "desc": "Full body myofascial release"}
#                 ]
#             elif workout_type == "Six-Pack Gain":
#                 recovery_activities = [
#                     {"name": "Core Stability Work", "duration": "20 min", "desc": "Light core activation"},
#                     {"name": "Light Cardio", "duration": "30 min", "desc": "Brisk walk or cycling"},
#                     {"name": "Mobility Drills", "duration": "15 min", "desc": "Joint mobility and stretching"}
#                 ]
#             else:  # Muscle Gain recovery
#                 recovery_activities = [
#                     {"name": "Light Cardio", "duration": "20-30 min", "desc": "Walking or cycling"},
#                     {"name": "Stretching", "duration": "10-15 min", "desc": "Full body stretching"},
#                     {"name": "Foam Rolling", "duration": "10 min", "desc": "Major muscle groups"}
#                 ]
            
#             for i, activity in enumerate(recovery_activities):
#                 data.append({
#                     "workout_day": day["day_name"],
#                     "day_number": day_num,
#                     "exercise_name": activity["name"],
#                     "stage_of_exercises": "recovery",
#                     "sequence_order": i + 1,
#                     "sets": 1,
#                     "reps": activity["duration"],
#                     "weight": "N/A",
#                     "rest_seconds": 0,
#                     "notes": f"{workout_type} - Recovery: {activity['desc']}",
#                     "workout_type": workout_type
#                 })
    
#     return pd.DataFrame(data)

# def generate_all_templates():
#     """
#     Generate template data for all workout types
#     """
#     templates_data = [
#         {
#             "id": str(uuid.uuid4()),
#             "name": "Muscle Gain Weekly Plan",
#             "description": "4-day split workout plan focusing on muscle hypertrophy with compound movements",
#             "difficulty_level": "Intermediate",
#             "estimated_duration_minutes": 60,
#             "target_muscle_groups": ["Chest", "Back", "Legs", "Shoulders", "Arms"],
#             "created_at": datetime.now().isoformat(),
#             "workout_type": "Muscle Gain"
#         },
#         {
#             "id": str(uuid.uuid4()),
#             "name": "Weight Loss Fat Burn Plan",
#             "description": "High-intensity cardio and circuit training focused on calorie burn and metabolic boost",
#             "difficulty_level": "Intermediate",
#             "estimated_duration_minutes": 45,
#             "target_muscle_groups": ["Full Body", "Core", "Cardiovascular"],
#             "created_at": datetime.now().isoformat(),
#             "workout_type": "Weight Loss"
#         },
#         {
#             "id": str(uuid.uuid4()),
#             "name": "Six-Pack Abs Development Plan",
#             "description": "Core-focused workout plan targeting abdominal definition and core strength",
#             "difficulty_level": "Intermediate-Advanced",
#             "estimated_duration_minutes": 50,
#             "target_muscle_groups": ["Abs", "Obliques", "Core", "Full Body"],
#             "created_at": datetime.now().isoformat(),
#             "workout_type": "Six-Pack Gain"
#         }
#     ]
    
#     return pd.DataFrame(templates_data)

# def save_workout_plans():
#     """
#     Save all workout plans to CSV files
#     """
#     print("🏋️‍♂️ Generating All Workout Plans...")
#     print("=" * 60)
    
#     # Generate workout plans
#     muscle_gain_df = generate_muscle_gain_workout()
#     weight_loss_df = generate_weight_loss_workout()
#     sixpack_df = generate_sixpack_workout()
    
#     # Combine all plans
#     all_plans_df = pd.concat([muscle_gain_df, weight_loss_df, sixpack_df], ignore_index=True)
    
#     # Generate templates
#     templates_df = generate_all_templates()
    
#     # Save individual files
#     muscle_gain_df.to_csv("muscle_gain_workout_plan.csv", index=False)
#     weight_loss_df.to_csv("weight_loss_workout_plan.csv", index=False)
#     sixpack_df.to_csv("sixpack_workout_plan.csv", index=False)
#     all_plans_df.to_csv("all_workout_plans.csv", index=False)
#     templates_df.to_csv("workout_templates.csv", index=False)
    
#     print("✅ All workout plans generated successfully!")
#     print("\n📁 Generated Files:")
#     print("  • muscle_gain_workout_plan.csv")
#     print("  • weight_loss_workout_plan.csv")
#     print("  • sixpack_workout_plan.csv")
#     print("  • all_workout_plans.csv")
#     print("  • workout_templates.csv")
    
#     # Print summaries
#     print("\n📊 Workout Plan Statistics:")
#     print("=" * 60)
    
#     for workout_type, df in [("Muscle Gain", muscle_gain_df), 
#                              ("Weight Loss", weight_loss_df), 
#                              ("Six-Pack Gain", sixpack_df)]:
#         print(f"\n{workout_type}:")
#         print(f"  Total exercises: {len(df)}")
#         print(f"  Workout days: {df[df['stage_of_exercises'].isin(['warmup', 'main', 'relax'])]['day_number'].nunique()}")
#         print(f"  Recovery days: {df[df['stage_of_exercises'] == 'recovery']['day_number'].nunique()}")
        
#         warmup_count = len(df[df['stage_of_exercises'] == 'warmup'])
#         main_count = len(df[df['stage_of_exercises'] == 'main'])
#         relax_count = len(df[df['stage_of_exercises'] == 'relax'])
#         recovery_count = len(df[df['stage_of_exercises'] == 'recovery'])
        
#         print(f"  Warmup exercises: {warmup_count}")
#         print(f"  Main exercises: {main_count}")
#         print(f"  Relax exercises: {relax_count}")
#         print(f"  Recovery activities: {recovery_count}")
    
#     return all_plans_df, templates_df

# def create_sample_template_exercises():
#     """
#     Create sample template_exercises data matching the database structure
#     """
#     print("\n🎯 Creating Template Exercise Samples...")
#     print("=" * 60)
    
#     sample_data = []
#     template_ids = []
    
#     # Get template IDs from generated templates
#     import csv
#     try:
#         with open('workout_templates.csv', 'r') as f:
#             reader = csv.DictReader(f)
#             for row in reader:
#                 template_ids.append(row['id'])
#     except:
#         # Create sample template IDs if file doesn't exist
#         template_ids = [str(uuid.uuid4()) for _ in range(3)]
    
#     # Sample exercises for each template
#     sample_exercises = [
#         {"name": "Bench Press", "muscle_group": "Chest"},
#         {"name": "Squats", "muscle_group": "Legs"},
#         {"name": "Pull-ups", "muscle_group": "Back"},
#         {"name": "Plank", "muscle_group": "Core"},
#         {"name": "Burpees", "muscle_group": "Full Body"},
#         {"name": "Russian Twists", "muscle_group": "Abs"}
#     ]
    
#     for template_idx, template_id in enumerate(template_ids[:3]):
#         workout_type = ["Muscle Gain", "Weight Loss", "Six-Pack Gain"][template_idx]
        
#         for day_num in range(1, 6):
#             if day_num % 2 == 0:  # Recovery days
#                 continue
                
#             # Add warmup, main, relax exercises
#             stages = ["warmup", "main", "relax"]
            
#             for stage_idx, stage in enumerate(stages):
#                 for ex_idx in range(3):  # 3 exercises per stage
#                     sample_data.append({
#                         "id": str(uuid.uuid4()),
#                         "template_id": template_id,
#                         "exercise_id": str(uuid.uuid4()),
#                         "stage_of_exercises": stage,
#                         "day_number": day_num,
#                         "sequence_order": stage_idx * 3 + ex_idx + 1,
#                         "sets": 3 if stage == "main" else 2,
#                         "reps": 12 if stage == "main" else 10,
#                         "weight": 0.0,
#                         "rest": 60 if stage == "main" else 30,
#                         "created_at": datetime.now().isoformat(),
#                         "workout_type": workout_type,
#                         "exercise_name": sample_exercises[ex_idx]["name"]
#                     })
    
#     sample_df = pd.DataFrame(sample_data)
#     sample_df.to_csv("template_exercises_sample.csv", index=False)
#     print("✅ Sample template_exercises saved to 'template_exercises_sample.csv'")
    
#     return sample_df

# def main():
#     """
#     Main function to generate all workout plans
#     """
#     print("🏆 COMPLETE WORKOUT PLAN GENERATOR 🏆")
#     print("=" * 60)
    
#     # Generate and save all workout plans
#     all_plans_df, templates_df = save_workout_plans()
    
#     # Create sample template exercises
#     template_exercises_df = create_sample_template_exercises()
    
#     # Display sample data
#     print("\n👀 Sample from Muscle Gain Plan (First 10 rows):")
#     print("=" * 60)
#     muscle_sample = pd.read_csv("muscle_gain_workout_plan.csv")
#     print(muscle_sample.head(10))
    
#     print("\n👀 Sample from Weight Loss Plan (First 10 rows):")
#     print("=" * 60)
#     weight_sample = pd.read_csv("weight_loss_workout_plan.csv")
#     print(weight_sample.head(10))
    
#     print("\n👀 Sample from Six-Pack Plan (First 10 rows):")
#     print("=" * 60)
#     sixpack_sample = pd.read_csv("sixpack_workout_plan.csv")
#     print(sixpack_sample.head(10))
    
#     print("\n🎉 All workout plans have been generated successfully!")
#     print("\n💡 Tips for using these plans:")
#     print("  1. Start with lighter weights and focus on form")
#     print("  2. Gradually increase intensity over weeks")
#     print("  3. Stay consistent and track your progress")
#     print("  4. Adjust based on your individual needs and recovery")
    
#     # Return dataframes for further processing if needed
#     return all_plans_df, templates_df, template_exercises_df

# if __name__ == "__main__":
#     # Run the main function
#     all_plans, templates, template_exercises = main()

In [12]:
templates_df = pd.read_csv("workout_templates.csv")
templates_df.head()

,id,name,description,difficulty_level,estimated_duration_minutes,target_muscle_groups,created_at,workout_type
0,6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b,Muscle Gain Weekly Plan,4-day split workout plan focusing on muscle hy...,Intermediate,60,"['Chest', 'Back', 'Legs', 'Shoulders', 'Arms']",2026-01-02T14:11:08.621479,Muscle Gain
1,666b7014-b672-4a13-8571-c885d96ef2de,Weight Loss Fat Burn Plan,High-intensity cardio and circuit training foc...,Intermediate,45,"['Full Body', 'Core', 'Cardiovascular']",2026-01-02T14:11:08.621479,Weight Loss
2,a20edc23-dea2-41d7-aff0-49ae738ec7b0,Six-Pack Abs Development Plan,Core-focused workout plan targeting abdominal ...,Intermediate-Advanced,50,"['Abs', 'Obliques', 'Core', 'Full Body']",2026-01-02T14:11:08.621479,Six-Pack Gain


In [31]:
templates_df.columns

Index(['id', 'name', 'description', 'difficulty_level',
       'estimated_duration_minutes', 'target_muscle_groups', 'created_at',
       'workout_type'],
      dtype='object')

In [ ]:


    # id = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    # name = Column(String(255), nullable=False, index=True)
    # description = Column(Text)
    # difficulty_level = Column(String(50))
    # estimated_duration_minutes = Column(Integer)
    # target_muscle_groups = Column(ARRAY(String))
    # workout_type = Column(String(50)) # warmup , main exercise, relex

    # created_at = Column(DateTime(timezone=True), server_default=func.now())
    # updated_at = Column(DateTime(timezone=True), onupdate=func.now())
    
    # # Relationships
    # template_exercises = relationship("TemplateExercise", back_populates="template")
    # user_workout_plans = relationship("UserWorkoutPlan", back_populates="template")


In [6]:
all_plans_df = pd.read_csv("all_workout_plans.csv")
all_plans_df.head(1)

,workout_day,day_number,exercise_name,stage_of_exercises,sequence_order,sets,reps,weight,rest_seconds,notes,workout_type
0,Day 1: Upper Body Push,1,Arm Circles,warmup,1,2,30 sec,Bodyweight,30,Muscle Gain - Warmup: Focus on form and mobility,Muscle Gain


In [8]:
all_plans_df['id'] = [str(uuid.uuid4()) for _ in range(len(all_plans_df))]

In [9]:
all_plans_df.head(1)

,workout_day,day_number,exercise_name,stage_of_exercises,sequence_order,sets,reps,weight,rest_seconds,notes,workout_type,id
0,Day 1: Upper Body Push,1,Arm Circles,warmup,1,2,30 sec,Bodyweight,30,Muscle Gain - Warmup: Focus on form and mobility,Muscle Gain,bce247d8-165a-42bb-8612-338c3b719000


In [14]:
def workout_id_fun(x):
    if x=="Muscle Gain": return "6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b"
    elif x=="Weight Loss": return "666b7014-b672-4a13-8571-c885d96ef2de"
    else: return "a20edc23-dea2-41d7-aff0-49ae738ec7b0"

all_plans_df['template_id'] = all_plans_df['workout_type'].apply(workout_id_fun)

In [15]:
all_plans_df.head(1)

,workout_day,day_number,exercise_name,stage_of_exercises,sequence_order,sets,reps,weight,rest_seconds,notes,workout_type,id,template_id
0,Day 1: Upper Body Push,1,Arm Circles,warmup,1,2,30 sec,Bodyweight,30,Muscle Gain - Warmup: Focus on form and mobility,Muscle Gain,bce247d8-165a-42bb-8612-338c3b719000,6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b


In [ ]:
# all_plans_df.to_csv("test_workout.csv", index=False)

In [1]:
import pandas as pd
workout = pd.read_csv("temp_workout - test_workout.csv")
workout.head(3)

,workout_day,day_number,exercise_name,stage_of_exercises,sequence_order,sets,reps,weight,rest_seconds,notes,workout_type,id,template_id,exercise_id
0,Day 1: Upper Body Push,1,Arm Circles,warmup,1,2,30 sec,Bodyweight,30,Muscle Gain - Warmup: Focus on form and mobility,Muscle Gain,bce247d8-165a-42bb-8612-338c3b719000,6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b,5b10c77c-b9d3-563c-8feb-9820c4efea7f
1,Day 1: Upper Body Push,1,Jumping Jacks,warmup,2,2,30 sec,Bodyweight,30,Muscle Gain - Warmup: Focus on form and mobility,Muscle Gain,9fbe9c58-1680-45d1-b092-90e892a4007a,6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b,7dded29d-b5ca-550c-8646-2d73b8f974cd
2,Day 1: Upper Body Push,1,Cat-Cow Stretch,warmup,3,2,10,Bodyweight,30,Muscle Gain - Warmup: Focus on form and mobility,Muscle Gain,412df284-f9aa-4415-9b6b-b69a8c479293,6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b,cc66e9b8-d656-514d-8a85-d497540f034d


In [2]:
workout.columns

Index(['workout_day', 'day_number', 'exercise_name', 'stage_of_exercises',
       'sequence_order', 'sets', 'reps', 'weight', 'rest_seconds', 'notes',
       'workout_type', 'id', 'template_id', 'exercise_id'],
      dtype='object')

In [10]:
all_plans_df.columns

Index(['workout_day', 'day_number', 'exercise_name', 'stage_of_exercises',
       'sequence_order', 'sets', 'reps', 'weight', 'rest_seconds', 'notes',
       'workout_type', 'id'],
      dtype='object')

In [ ]:

# main =  ['workout_day', 'day_number', 'exercise_name', 'stage_of_exercises', 'sequence_order', 'sets', 'reps', 'weight', 'rest_seconds', 'notes','workout_type', 'id', 'template_id', 'exercise_id']

    # id = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    # template_id = Column(UUID(as_uuid=True), ForeignKey("workout_templates.id"), nullable=False, index=True)
    # exercise_id = Column(UUID(as_uuid=True), ForeignKey("exercises.id"), nullable=False, index=True)

    # workout_day = Column(String(50)) # warmup , main exercise, relex
    # day_number = Column(Integer, nullable=False)  # 1, 2, 3 for 3-day split
    # exercise_name = Column(String(50)) # warmup , main exercise, relex
    # stage_of_exercises = Column(String(50)) # warmup , main exercise, relex
    # sequence_order = Column(Integer, nullable=False)
    # sets = Column(Integer)
    # reps = Column(String(50))
    # weight = Column(Float)
    # rest_seconds = Column(Integer)
    # notes = Column(Text) # warmup , main exercise, relex
    # workout_type = Column(String(50)) 

    # created_at = Column(DateTime(timezone=True), server_default=func.now())
    # updated_at = Column(DateTime(timezone=True), onupdate=func.now())
    
    # # Relationships
    # template = relationship("WorkoutTemplate", back_populates="template_exercises")
    # exercise = relationship("Exercise", back_populates="template_exercises")



In [2]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from io import StringIO
import time
import uuid
import ast

def ultra_fast_csv_to_workout_templates(csv_path, table_name="workout_templates"):
    """
    Ultra-fast CSV to PostgreSQL for workout_templates table
    """
    
    print("⚡ ULTRA-FAST CSV IMPORT - WORKOUT TEMPLATES")
    print("="*60)
    
    start_time = time.time()
    
    # Database config
    db_config = {
        'host': 'localhost',
        'database': 'GoLift_db',
        'user': 'postgres',
        'password': 'kali7',
        'port': '5432'
    }
    
    try:
        # Step 1: Load CSV
        csv_start = time.time()
        df = pd.read_csv(csv_path)
        csv_time = time.time() - csv_start
        print(f"✅ CSV loaded: {len(df)} rows in {csv_time:.2f}s")
        print(f"📋 Columns: {list(df.columns)}")
        
        # Step 2: Ensure all required columns exist
        required_columns = ['name']
        for col in required_columns:
            if col not in df.columns:
                raise ValueError(f"Missing required column: {col}")
        
        # Step 3: Handle ID generation
        print("🔄 Processing IDs...")
        id_start = time.time()
        
        # Generate UUIDs for missing IDs
        if 'id' not in df.columns:
            print("  Generating new UUIDs...")
            df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]
        else:
            # Check if IDs are numeric or need conversion
            try:
                # Try to convert to UUID if they're numeric
                numeric_ids = pd.to_numeric(df['id'], errors='coerce')
                if numeric_ids.notna().all():
                    # Convert numeric IDs to UUIDs
                    def numeric_id_to_uuid(numeric_id):
                        namespace = uuid.UUID('6ba7b810-9dad-11d1-80b4-00c04fd430c8')
                        return str(uuid.uuid5(namespace, str(numeric_id)))
                    
                    df['id'] = numeric_ids.apply(numeric_id_to_uuid)
                    print(f"  Converted {len(df)} numeric IDs to UUIDs")
                else:
                    # IDs are already strings/UUIDs, ensure they're valid UUID format
                    try:
                        df['id'] = df['id'].apply(lambda x: str(uuid.UUID(str(x))) if pd.notna(x) else str(uuid.uuid4()))
                        print("  Validated UUID formats")
                    except:
                        print("  IDs are in string format (not UUID)")
            except:
                # If conversion fails, generate new UUIDs
                print("  Generating new UUIDs for invalid IDs...")
                df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]
        
        id_time = time.time() - id_start
        print(f"✅ ID processing: {id_time:.2f}s")
        
        # Step 4: Process array columns
        print("🔄 Processing array columns...")
        array_start = time.time()
        
        # Handle target_muscle_groups array
        if 'target_muscle_groups' in df.columns:
            def parse_array(value):
                if pd.isna(value):
                    return None
                # Try to parse as Python list if it looks like one
                if isinstance(value, str) and value.startswith('[') and value.endswith(']'):
                    try:
                        # Safely evaluate string as Python list
                        parsed = ast.literal_eval(value)
                        if isinstance(parsed, list):
                            return parsed
                    except:
                        pass
                # If it's a comma-separated string
                if isinstance(value, str):
                    return [item.strip() for item in value.split(',') if item.strip()]
                # Return as single element array
                return [str(value)]
            
            df['target_muscle_groups'] = df['target_muscle_groups'].apply(parse_array)
            print(f"  Processed target_muscle_groups as array")
        
        array_time = time.time() - array_start
        print(f"✅ Array processing: {array_time:.2f}s")
        
        # Step 5: Clean and prepare data
        print("🔄 Cleaning data...")
        clean_start = time.time()
        
        # Ensure string columns don't exceed max length
        if 'name' in df.columns:
            df['name'] = df['name'].str.slice(0, 255)
        
        if 'workout_type' in df.columns:
            df['workout_type'] = df['workout_type'].str.slice(0, 50)
        
        if 'difficulty_level' in df.columns:
            df['difficulty_level'] = df['difficulty_level'].str.slice(0, 50)
        
        # Ensure estimated_duration_minutes is integer
        if 'estimated_duration_minutes' in df.columns:
            df['estimated_duration_minutes'] = pd.to_numeric(
                df['estimated_duration_minutes'], 
                errors='coerce'
            ).astype('Int64')  # Allows NaN
        
        clean_time = time.time() - clean_start
        print(f"✅ Data cleaning: {clean_time:.2f}s")
        
        # Step 6: Connect and use COPY
        copy_start = time.time()
        
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Create temporary staging table
        staging_table = f"staging_{table_name}_{int(time.time())}"
        
        # Create staging table with TEXT columns (for flexible import)
        cursor.execute(sql.SQL("""
            CREATE TEMP TABLE {staging_table} (
                id TEXT,
                name TEXT,
                description TEXT,
                difficulty_level TEXT,
                estimated_duration_minutes TEXT,
                target_muscle_groups TEXT,
                workout_type TEXT
            ) ON COMMIT DROP
        """).format(staging_table=sql.Identifier(staging_table)))
        
        # Prepare data for COPY
        output = StringIO()
        
        # Write to CSV format without headers
        # We'll use tab as delimiter and \\N for NULL
        for _, row in df.iterrows():
            row_str = []
            for val in row:
                # FIXED: Handle different types properly
                if isinstance(val, list):
                    # Convert list to PostgreSQL array string format
                    # Escape special characters in each element
                    escaped_items = []
                    for item in val:
                        if item is None or pd.isna(item):
                            continue
                        item_str = str(item)
                        item_str = item_str.replace('\\', '\\\\')
                        item_str = item_str.replace('"', '""')
                        escaped_items.append(f'"{item_str}"')
                    
                    if escaped_items:
                        array_str = "{" + ",".join(escaped_items) + "}"
                        row_str.append(array_str)
                    else:
                        row_str.append('\\N')
                elif pd.isna(val):
                    row_str.append('\\N')
                else:
                    val_str = str(val)
                    val_str = val_str.replace('\\', '\\\\')
                    val_str = val_str.replace('\t', '\\t')
                    val_str = val_str.replace('\n', '\\n')
                    val_str = val_str.replace('\r', '\\r')
                    row_str.append(val_str)
            
            output.write('\t'.join(row_str) + '\n')
        
        output.seek(0)
        
        # Copy to staging table
        cursor.copy_from(output, staging_table, null='\\N', sep='\t')
        print(f"✅ Data copied to staging table")
        
        # Step 7: Backup existing data
        backup_start = time.time()
        timestamp = int(time.time())
        backup_table = f"{table_name}_backup_{timestamp}"
        
        cursor.execute(sql.SQL("""
            CREATE TABLE {backup_table} AS 
            SELECT * FROM {table_name} WHERE 1=0
        """).format(
            backup_table=sql.Identifier(backup_table),
            table_name=sql.Identifier(table_name)
        ))
        
        # Copy existing data to backup
        cursor.execute(sql.SQL("""
            INSERT INTO {backup_table}
            SELECT * FROM {table_name}
        """).format(
            backup_table=sql.Identifier(backup_table),
            table_name=sql.Identifier(table_name)
        ))
        backup_time = time.time() - backup_start
        print(f"✅ Backup created: {backup_table} ({cursor.rowcount} rows)")
        
        # Step 8: Clear and insert from staging
        insert_start = time.time()
        
        # Clear existing data
        cursor.execute(sql.SQL("TRUNCATE TABLE {table_name} CASCADE").format(
            table_name=sql.Identifier(table_name)
        ))
        
        # Insert from staging with proper type casting
        cursor.execute(sql.SQL("""
            INSERT INTO {table_name} (
                id, name, description, difficulty_level, 
                estimated_duration_minutes, target_muscle_groups, 
                workout_type, created_at
            )
            SELECT 
                s.id::UUID,
                s.name,
                NULLIF(TRIM(s.description), ''),
                NULLIF(TRIM(s.difficulty_level), ''),
                NULLIF(s.estimated_duration_minutes, '')::INTEGER,
                CASE 
                    WHEN s.target_muscle_groups IS NULL OR TRIM(s.target_muscle_groups) = '' 
                    THEN NULL 
                    ELSE s.target_muscle_groups::VARCHAR[] 
                END,
                NULLIF(TRIM(s.workout_type), ''),
                CURRENT_TIMESTAMP
            FROM {staging_table} s
        """).format(
            table_name=sql.Identifier(table_name),
            staging_table=sql.Identifier(staging_table)
        ))
        
        insert_time = time.time() - insert_start
        inserted_count = cursor.rowcount
        print(f"✅ Inserted {inserted_count} rows")
        
        conn.commit()
        cursor.close()
        conn.close()
        
        copy_time = time.time() - copy_start
        
        total_time = time.time() - start_time
        
        print("\n" + "="*60)
        print("📈 PERFORMANCE RESULTS")
        print("="*60)
        print(f"📥 CSV Load:          {csv_time:.3f} seconds")
        print(f"🔄 ID Processing:     {id_time:.3f} seconds")
        print(f"📊 Array Processing:  {array_time:.3f} seconds")
        print(f"🧹 Data Cleaning:     {clean_time:.3f} seconds")
        print(f"💾 Backup:            {backup_time:.3f} seconds")
        print(f"📤 Data Insert:       {insert_time:.3f} seconds")
        print("-"*60)
        print(f"🚀 TOTAL TIME:        {total_time:.2f} seconds")
        print(f"📊 Rows processed:    {inserted_count}")
        print(f"📈 Speed:             {inserted_count/total_time:.0f} rows/second")
        
        # Verify
        if inserted_count == len(df):
            print("✅ SUCCESS: All rows inserted correctly!")
        else:
            print(f"⚠️  WARNING: Expected {len(df)} rows, got {inserted_count}")
        
        print(f"💾 Backup table: {backup_table}")
        
        return df, total_time
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        raise

# Alternative simpler version using StringIO with proper formatting
def simple_csv_to_workout_templates(csv_path, table_name="workout_templates"):
    """
    Simpler alternative that handles arrays properly
    """
    print("🔄 SIMPLER CSV IMPORT - WORKOUT TEMPLATES")
    print("="*60)
    
    start_time = time.time()
    
    # Database config
    db_config = {
        'host': 'localhost',
        'database': 'GoLift_db',
        'user': 'postgres',
        'password': 'kali7',
        'port': '5432'
    }
    
    try:
        # Load CSV
        df = pd.read_csv(csv_path)
        print(f"✅ CSV loaded: {len(df)} rows")
        
        # Process IDs
        if 'id' not in df.columns:
            df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]
        
        # Process array column
        if 'target_muscle_groups' in df.columns:
            def safe_parse_array(val):
                if pd.isna(val):
                    return None
                if isinstance(val, str):
                    # Remove brackets if present
                    if val.startswith('[') and val.endswith(']'):
                        val = val[1:-1]
                    # Split by comma and clean
                    items = [item.strip().strip('"\'') for item in val.split(',')]
                    # Remove empty strings
                    return [item for item in items if item]
                return [str(val)]
            
            df['target_muscle_groups'] = df['target_muscle_groups'].apply(safe_parse_array)
        
        # Connect to database
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Backup existing data
        backup_table = f"{table_name}_backup_{int(time.time())}"
        cursor.execute(sql.SQL("CREATE TABLE {backup} AS SELECT * FROM {table}").format(
            backup=sql.Identifier(backup_table),
            table=sql.Identifier(table_name)
        ))
        print(f"✅ Backup created: {backup_table}")
        
        # Clear table
        cursor.execute(sql.SQL("TRUNCATE TABLE {table} CASCADE").format(
            table=sql.Identifier(table_name)
        ))
        
        # Insert data using executemany (slower but simpler)
        inserted_count = 0
        for _, row in df.iterrows():
            # Prepare values
            values = {
                'id': str(uuid.UUID(row['id'])) if 'id' in row else str(uuid.uuid4()),
                'name': row.get('name', ''),
                'description': row.get('description', ''),
                'difficulty_level': row.get('difficulty_level', ''),
                'estimated_duration_minutes': row.get('estimated_duration_minutes'),
                'target_muscle_groups': row.get('target_muscle_groups'),
                'workout_type': row.get('workout_type', '')
            }
            
            # Insert
            cursor.execute(sql.SQL("""
                INSERT INTO {table} (
                    id, name, description, difficulty_level, 
                    estimated_duration_minutes, target_muscle_groups, 
                    workout_type, created_at
                ) VALUES (
                    %s, %s, %s, %s, %s, %s, %s, CURRENT_TIMESTAMP
                )
            """).format(table=sql.Identifier(table_name)), [
                values['id'],
                values['name'][:255] if values['name'] else None,
                values['description'] or None,
                values['difficulty_level'][:50] if values['difficulty_level'] else None,
                int(values['estimated_duration_minutes']) if pd.notna(values['estimated_duration_minutes']) else None,
                values['target_muscle_groups'],
                values['workout_type'][:50] if values['workout_type'] else None
            ])
            inserted_count += 1
        
        conn.commit()
        cursor.close()
        conn.close()
        
        total_time = time.time() - start_time
        print(f"✅ Inserted {inserted_count} rows in {total_time:.2f}s")
        
        return df, total_time
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        raise

# Run the function - try the simpler version if the fast one fails
try:
    df, total_time = ultra_fast_csv_to_workout_templates("workout_templates.csv")
except Exception as e:
    print(f"\n⚠️ Fast method failed: {e}")
    print("Trying simpler method...\n")
    df, total_time = simple_csv_to_workout_templates("workout_templates.csv")

⚡ ULTRA-FAST CSV IMPORT - WORKOUT TEMPLATES
✅ CSV loaded: 3 rows in 0.40s
📋 Columns: ['id', 'name', 'description', 'difficulty_level', 'estimated_duration_minutes', 'target_muscle_groups', 'created_at', 'workout_type']
🔄 Processing IDs...
  Validated UUID formats
✅ ID processing: 0.05s
🔄 Processing array columns...
  Processed target_muscle_groups as array
✅ Array processing: 0.00s
🔄 Cleaning data...
✅ Data cleaning: 0.02s
❌ Error: extra data after last expected column
CONTEXT:  COPY staging_workout_templates_1767358588, line 1: "6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b	Muscle Gain Weekly Plan	4-day split workout plan focusing on mu..."


⚠️ Fast method failed: extra data after last expected column
CONTEXT:  COPY staging_workout_templates_1767358588, line 1: "6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b	Muscle Gain Weekly Plan	4-day split workout plan focusing on mu..."

Trying simpler method...

🔄 SIMPLER CSV IMPORT - WORKOUT TEMPLATES
✅ CSV loaded: 3 rows
✅ Backup created: workout_templates_back

Traceback (most recent call last):
  File "C:\Users\pavan\AppData\Local\Temp\ipykernel_18116\271747156.py", line 195, in ultra_fast_csv_to_workout_templates
    cursor.copy_from(output, staging_table, null='\\N', sep='\t')
psycopg2.errors.BadCopyFileFormat: extra data after last expected column
CONTEXT:  COPY staging_workout_templates_1767358588, line 1: "6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b	Muscle Gain Weekly Plan	4-day split workout plan focusing on mu..."



✅ Inserted 3 rows in 1.07s


In [ ]:
# import pandas as pd
# import psycopg2
# from psycopg2 import sql
# from io import StringIO
# import time
# import uuid

# def ultra_fast_template_exercises_to_postgres(csv_path, table_name="template_exercises"):
#     """
#     Ultra-fast CSV to PostgreSQL for template_exercises table
#     Assumes CSV already contains UUIDs for id, template_id, exercise_id
#     """
    
#     print("⚡ ULTRA-FAST TEMPLATE EXERCISES CSV IMPORT")
#     print("="*60)
    
#     start_time = time.time()
    
#     # Database config
#     db_config = {
#         'host': 'localhost',
#         'database': 'GoLift_db',
#         'user': 'postgres',
#         'password': 'kali7',
#         'port': '5432'
#     }
    
#     try:
#         # Step 1: Load CSV
#         csv_start = time.time()
#         df = pd.read_csv(csv_path)
#         csv_time = time.time() - csv_start
#         print(f"✅ CSV loaded: {len(df)} rows in {csv_time:.2f}s")
        
#         # Display sample of ID columns to verify format
#         print("\n📋 Sample ID values (first 3 rows):")
#         id_columns = ['id', 'template_id', 'exercise_id']
#         for col in id_columns:
#             if col in df.columns:
#                 print(f"  {col}: {df[col].iloc[0] if len(df) > 0 else 'N/A'}")
        
#         # Step 2: No ID conversion needed - CSV already has UUIDs
#         print("✅ Using existing UUID values from CSV")
        
#         # Step 3: Check and ensure column order matches database
#         print("\n🔍 Checking column structure...")
        
#         # Database columns in expected order
#         db_columns = [
#             'id', 'template_id', 'exercise_id', 'workout_day', 'day_number',
#             'exercise_name', 'stage_of_exercises', 'sequence_order', 'sets',
#             'reps', 'weight', 'rest_seconds', 'notes', 'workout_type'
#         ]
        
#         # Check for missing columns
#         missing_columns = [col for col in db_columns if col not in df.columns]
#         if missing_columns:
#             print(f"⚠️  Warning: Missing columns in CSV: {missing_columns}")
#             # Add missing columns with None values
#             for col in missing_columns:
#                 df[col] = None
        
#         # Reorder columns to match database
#         df = df[db_columns]
#         print(f"✅ Columns reordered to match database schema")
        
#         # Step 4: Clean and validate data
#         print("\n🧹 Cleaning data...")
        
#         # Convert numeric columns, handling errors
#         numeric_columns = ['day_number', 'sequence_order', 'sets', 'rest_seconds']
#         for col in numeric_columns:
#             if col in df.columns:
#                 # Convert to numeric, coerce errors to NaN
#                 df[col] = pd.to_numeric(df[col], errors='coerce')
#                 # Count nulls after conversion
#                 null_count = df[col].isna().sum()
#                 if null_count > 0:
#                     print(f"  {col}: {null_count} non-numeric values converted to NULL")
        
#         # Convert weight to float
#         if 'weight' in df.columns:
#             df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
        
#         print("✅ Data cleaning complete")
        
#         # Step 5: Connect and use COPY
#         copy_start = time.time()
        
#         conn = psycopg2.connect(**db_config)
#         cursor = conn.cursor()
        
#         # Create temporary staging table
#         staging_table = f"staging_{table_name}_{int(time.time())}"
        
#         # Create staging table with TEXT columns
#         cursor.execute(sql.SQL("""
#             CREATE TEMP TABLE {staging_table} (
#                 id TEXT,
#                 template_id TEXT,
#                 exercise_id TEXT,
#                 workout_day TEXT,
#                 day_number TEXT,
#                 exercise_name TEXT,
#                 stage_of_exercises TEXT,
#                 sequence_order TEXT,
#                 sets TEXT,
#                 reps TEXT,
#                 weight TEXT,
#                 rest_seconds TEXT,
#                 notes TEXT,
#                 workout_type TEXT
#             ) ON COMMIT DROP
#         """).format(staging_table=sql.Identifier(staging_table)))
        
#         # Prepare data for COPY
#         output = StringIO()
        
#         # Write to CSV format without headers
#         for _, row in df.iterrows():
#             row_str = []
#             for val in row:
#                 if pd.isna(val) or val is None:
#                     row_str.append('\\N')  # PostgreSQL NULL
#                 else:
#                     # Convert to string and escape special characters
#                     val_str = str(val)
#                     # Escape special characters
#                     val_str = val_str.replace('\\', '\\\\')
#                     val_str = val_str.replace('\t', '\\t')
#                     val_str = val_str.replace('\n', '\\n')
#                     val_str = val_str.replace('\r', '\\r')
#                     row_str.append(val_str)
#             output.write('\t'.join(row_str) + '\n')
        
#         output.seek(0)
        
#         # Copy to staging table
#         cursor.copy_from(output, staging_table, null='\\N', sep='\t')
#         print(f"✅ Data copied to staging table: {len(df)} rows")
        
#         # Step 6: Backup existing data
#         backup_start = time.time()
#         timestamp = int(time.time())
#         backup_table = f"{table_name}_backup_{timestamp}"
        
#         cursor.execute(sql.SQL("""
#             CREATE TABLE {backup_table} AS 
#             SELECT * FROM {table_name} WHERE 1=0
#         """).format(
#             backup_table=sql.Identifier(backup_table),
#             table_name=sql.Identifier(table_name)
#         ))
        
#         # Copy existing data to backup
#         cursor.execute(sql.SQL("""
#             INSERT INTO {backup_table}
#             SELECT * FROM {table_name}
#         """).format(
#             backup_table=sql.Identifier(backup_table),
#             table_name=sql.Identifier(table_name)
#         ))
#         backup_time = time.time() - backup_start
#         backup_rows = cursor.rowcount
#         print(f"✅ Backup created: {backup_table} ({backup_rows} rows)")
        
#         # Step 7: Clear and insert from staging
#         insert_start = time.time()
        
#         # Clear existing data
#         cursor.execute(sql.SQL("TRUNCATE TABLE {table_name} CASCADE").format(
#             table_name=sql.Identifier(table_name)
#         ))
        
#         # Insert from staging - using UUIDs directly
#         cursor.execute(sql.SQL("""
#             INSERT INTO {table_name} (
#                 id, template_id, exercise_id, workout_day, day_number,
#                 exercise_name, stage_of_exercises, sequence_order, sets,
#                 reps, weight, rest_seconds, notes, workout_type,
#                 created_at, updated_at
#             )
#             SELECT 
#                 s.id::UUID,
#                 s.template_id::UUID,
#                 s.exercise_id::UUID,
#                 NULLIF(TRIM(s.workout_day), ''),
#                 CASE 
#                     WHEN NULLIF(TRIM(s.day_number), '') ~ '^[0-9]+$' 
#                     THEN NULLIF(TRIM(s.day_number), '')::INTEGER 
#                     ELSE NULL 
#                 END,
#                 NULLIF(TRIM(s.exercise_name), ''),
#                 NULLIF(TRIM(s.stage_of_exercises), ''),
#                 CASE 
#                     WHEN NULLIF(TRIM(s.sequence_order), '') ~ '^[0-9]+$' 
#                     THEN NULLIF(TRIM(s.sequence_order), '')::INTEGER 
#                     ELSE NULL 
#                 END,
#                 CASE 
#                     WHEN NULLIF(TRIM(s.sets), '') ~ '^[0-9]+$' 
#                     THEN NULLIF(TRIM(s.sets), '')::INTEGER 
#                     ELSE NULL 
#                 END,
#                 NULLIF(TRIM(s.reps), ''),
#                 CASE 
#                     WHEN NULLIF(TRIM(s.weight), '') ~ '^[0-9]*\.?[0-9]+$' 
#                     THEN NULLIF(TRIM(s.weight), '')::FLOAT 
#                     ELSE NULL 
#                 END,
#                 CASE 
#                     WHEN NULLIF(TRIM(s.rest_seconds), '') ~ '^[0-9]+$' 
#                     THEN NULLIF(TRIM(s.rest_seconds), '')::INTEGER 
#                     ELSE NULL 
#                 END,
#                 NULLIF(TRIM(s.notes), ''),
#                 NULLIF(TRIM(s.workout_type), ''),
#                 CURRENT_TIMESTAMP,
#                 CURRENT_TIMESTAMP
#             FROM {staging_table} s
#             WHERE s.id IS NOT NULL 
#               AND s.id != '\\N'
#               AND s.template_id IS NOT NULL
#               AND s.template_id != '\\N'
#               AND s.exercise_id IS NOT NULL
#               AND s.exercise_id != '\\N'
#         """).format(
#             table_name=sql.Identifier(table_name),
#             staging_table=sql.Identifier(staging_table)
#         ))
        
#         insert_time = time.time() - insert_start
#         inserted_count = cursor.rowcount
#         print(f"✅ Inserted {inserted_count} rows")
        
#         # Check for any invalid rows that weren't inserted
#         cursor.execute(sql.SQL("""
#             SELECT COUNT(*) 
#             FROM {staging_table} s
#             WHERE s.id IS NULL OR s.id = '\\N'
#                OR s.template_id IS NULL OR s.template_id = '\\N'
#                OR s.exercise_id IS NULL OR s.exercise_id = '\\N'
#         """).format(staging_table=sql.Identifier(staging_table)))
        
#         invalid_rows = cursor.fetchone()[0]
#         if invalid_rows > 0:
#             print(f"⚠️  Skipped {invalid_rows} rows with missing/invalid IDs")
        
#         conn.commit()
#         cursor.close()
#         conn.close()
        
#         copy_time = time.time() - copy_start
        
#         total_time = time.time() - start_time
        
#         print("\n" + "="*60)
#         print("📈 IMPORT SUMMARY")
#         print("="*60)
#         print(f"📥 CSV Load:           {csv_time:.3f} seconds")
#         print(f"🧹 Data Cleaning:      {insert_start - copy_start - csv_time:.3f} seconds")
#         print(f"💾 Backup:             {backup_time:.3f} seconds ({backup_rows} rows)")
#         print(f"📤 Data Insert:        {insert_time:.3f} seconds")
#         print("-"*60)
#         print(f"🚀 TOTAL TIME:         {total_time:.2f} seconds")
#         print(f"📊 Rows processed:     {inserted_count} of {len(df)}")
#         if invalid_rows > 0:
#             print(f"📊 Rows skipped:       {invalid_rows}")
#         print(f"📈 Speed:              {inserted_count/total_time:.0f} rows/second")
        
#         # Success message
#         if inserted_count == len(df):
#             print("\n🎉 SUCCESS: All rows imported successfully!")
#         elif inserted_count > 0:
#             print(f"\n✅ PARTIAL SUCCESS: {inserted_count} of {len(df)} rows imported")
#         else:
#             print("\n❌ IMPORT FAILED: No rows were imported")
        
#         print(f"💾 Backup table: {backup_table}")
        
#         # Display sample of imported data
#         print("\n👀 Sample of imported data (first 3 rows):")
#         for i in range(min(3, len(df))):
#             print(f"\nRow {i+1}:")
#             print(f"  ID: {df['id'].iloc[i]}")
#             print(f"  Template ID: {df['template_id'].iloc[i]}")
#             print(f"  Exercise ID: {df['exercise_id'].iloc[i]}")
#             print(f"  Exercise: {df['exercise_name'].iloc[i] if pd.notna(df['exercise_name'].iloc[i]) else 'N/A'}")
#             print(f"  Day: {df['day_number'].iloc[i] if pd.notna(df['day_number'].iloc[i]) else 'N/A'}")
        
#         return df, total_time
        
#     except Exception as e:
#         print(f"❌ Error: {e}")
#         import traceback
#         traceback.print_exc()
        
#         # Try to provide more specific error info
#         if "invalid input syntax for type uuid" in str(e):
#             print("\n🔍 UUID FORMAT ERROR DETECTED")
#             print("Please check that your CSV contains valid UUIDs in these formats:")
#             print("1. With hyphens: 123e4567-e89b-12d3-a456-426614174000")
#             print("2. Without hyphens: 123e4567e89b12d3a456426614174000")
#             print("3. Uppercase or lowercase both work")
        
#         raise


# def quick_import_template_exercises(csv_path, table_name="template_exercises"):
#     """
#     Simplified version for quick imports
#     """
#     print("🚀 QUICK IMPORT - TEMPLATE EXERCISES")
#     print("="*60)
    
#     try:
#         # Load CSV
#         df = pd.read_csv(csv_path)
#         print(f"📊 CSV loaded: {len(df)} rows")
        
#         # Check required columns
#         required = ['id', 'template_id', 'exercise_id']
#         missing = [col for col in required if col not in df.columns]
#         if missing:
#             print(f"❌ Missing columns: {missing}")
#             return None
        
#         # Database config
#         db_config = {
#             'host': 'localhost',
#             'database': 'GoLift_db',
#             'user': 'postgres',
#             'password': 'kali7',
#             'port': '5432'
#         }
        
#         # Connect
#         conn = psycopg2.connect(**db_config)
#         cursor = conn.cursor()
        
#         # Backup count
#         cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
#         existing_count = cursor.fetchone()[0]
#         print(f"📊 Existing rows in database: {existing_count}")
        
#         # Create staging table
#         staging = f"staging_{int(time.time())}"
#         cursor.execute(f"""
#             CREATE TEMP TABLE {staging} AS 
#             SELECT * FROM {table_name} WHERE 1=0
#         """)
        
#         # Simple insert
#         inserted = 0
#         for _, row in df.iterrows():
#             try:
#                 cursor.execute(f"""
#                     INSERT INTO {table_name} (
#                         id, template_id, exercise_id, workout_day, day_number,
#                         exercise_name, stage_of_exercises, sequence_order, sets,
#                         reps, weight, rest_seconds, notes, workout_type,
#                         created_at, updated_at
#                     ) VALUES (
#                         %s::UUID, %s::UUID, %s::UUID, %s, %s,
#                         %s, %s, %s, %s, %s,
#                         %s, %s, %s, %s,
#                         CURRENT_TIMESTAMP, CURRENT_TIMESTAMP
#                     )
#                 """, (
#                     row.get('id'), row.get('template_id'), row.get('exercise_id'),
#                     row.get('workout_day'), row.get('day_number'),
#                     row.get('exercise_name'), row.get('stage_of_exercises'), 
#                     row.get('sequence_order'), row.get('sets'),
#                     row.get('reps'), row.get('weight'), row.get('rest_seconds'),
#                     row.get('notes'), row.get('workout_type')
#                 ))
#                 inserted += 1
#             except Exception as e:
#                 print(f"⚠️  Skipping row: {e}")
#                 continue
        
#         conn.commit()
#         cursor.close()
#         conn.close()
        
#         print(f"✅ Inserted {inserted} of {len(df)} rows")
#         return inserted
        
#     except Exception as e:
#         print(f"❌ Error: {e}")
#         return None


# # Main execution
# if __name__ == "__main__":
#     # Update this path to your CSV file
#     csv_file_path = "temp_workout - test_workout.csv"  # Update this path
    
#     print("TEMPLATE EXERCISES CSV IMPORT")
#     print("="*60)
    
#     # Option 1: Use the ultra-fast method (recommended for large files)
#     print("\nMethod 1: Ultra-fast import (recommended)")
#     try:
#         df, total_time = ultra_fast_template_exercises_to_postgres(csv_file_path)
#         print(f"\n🎉 Import completed in {total_time:.2f} seconds!")
#     except Exception as e:
#         print(f"\n❌ Ultra-fast import failed: {e}")
        
#         # Option 2: Try the simple method
#         print("\n" + "="*60)
#         print("Method 2: Trying simple import...")
#         result = quick_import_template_exercises(csv_file_path)
#         if result:
#             print(f"\n✅ Simple import completed: {result} rows inserted")
#         else:
#             print("\n❌ Both import methods failed")

TEMPLATE EXERCISES CSV IMPORT

Method 1: Ultra-fast import (recommended)
⚡ ULTRA-FAST TEMPLATE EXERCISES CSV IMPORT
✅ CSV loaded: 159 rows in 0.39s

📋 Sample ID values (first 3 rows):
  id: bce247d8-165a-42bb-8612-338c3b719000
  template_id: 6fb6f05b-e9b3-49b2-afb7-ac99187a2e7b
  exercise_id: 5b10c77c-b9d3-563c-8feb-9820c4efea7f
✅ Using existing UUID values from CSV

🔍 Checking column structure...
✅ Columns reordered to match database schema

🧹 Cleaning data...
✅ Data cleaning complete
✅ Data copied to staging table: 159 rows
✅ Backup created: template_exercises_backup_1767502966 (0 rows)
✅ Inserted 159 rows

📈 IMPORT SUMMARY
📥 CSV Load:           0.395 seconds
🧹 Data Cleaning:      0.270 seconds
💾 Backup:             0.025 seconds (0 rows)
📤 Data Insert:        0.173 seconds
------------------------------------------------------------
🚀 TOTAL TIME:         1.45 seconds
📊 Rows processed:     159 of 159
📈 Speed:              109 rows/second

🎉 SUCCESS: All rows imported successfully!
💾 